## Seventh step

In `seventh_step.ipynb`, we will use the records filtered (or not) in `sixth_step`.ipynb and filter the herbarium specimen images for subsequent use in an artificial intelligence system, at `pipeline_classifier/separating.ipynb`.

In [ ]:
from library import *

specieslink, db_config = configure()

In [ ]:
conn = mysql_conn.connect(**db_config)
cursor = conn.cursor()

requirements = "country_upd"
requirements2 = "stateprovince_upd"
requirements3 = "barcode_upd"
requirements4 = "identifiedby_upd"
column = "barcode"
table = "biodiversity_records"

sql = f"""SELECT {column} FROM {table} WHERE {column} IS NOT NULL AND {requirements} IS NOT NULL AND {requirements2} IS NOT NULL AND {requirements3} IS NOT NULL AND {requirements4} IS NOT NULL"""

cursor.execute(sql)
results = cursor.fetchall()

for result in results:
    print(result[0])

cursor.close()
conn.close()

barcodes = [r[0] for r in results if r[0]]

df = pd.DataFrame({"barcode": barcodes})

with tempfile.NamedTemporaryFile(
    mode="w",
    suffix=".csv",
    delete=False,
    encoding="utf-8"
) as tmp:
    df.to_csv(tmp.name, index=False)
    csv_path = tmp.name

In [ ]:
family = input("enter the plant family name to save the output file.: ").strip()
csv = csv_path

script_path = os.path.abspath(os.path.join('..', 'herbcore_tool', 'downloader-specieslink-master', 'main.py'))
working_dir = os.path.dirname(script_path)
pipeline_dir = os.getcwd()

files_before = set(os.listdir(working_dir))
try:
    print(f"\nexecuting crawler for the family '{family}' to collect their URLs...\n")
    print("\nit's normal that it takes a long time !!!\n")
    process = subprocess.Popen(
            ['python', script_path, '--family', family, '--csv', os.path.abspath(csv)],
            cwd=working_dir,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            universal_newlines=True
        )

    for line in process.stdout:
        print(f"{line.strip()}")
    
    process.wait() 
        
    if process.returncode == 0:
        files_later = set(os.listdir(working_dir))
        new_files = files_later - files_before

        print(f"new files detected in the crawler's folder: {new_files}")
        found_file = False
        for file in new_files:
            print(f"analyzing file: {file}")
            
            if family.lower() in file.lower() and file.lower().endswith('.csv'):
                origin_path = os.path.join(working_dir, file)
                destiny_path = os.path.join(pipeline_dir, file)
                
                shutil.move(origin_path, destiny_path)
                print(f"moved: {file}")
                found_file = True
                break

        if not found_file:
            print(f"current content in the crawler's folder: {os.listdir(working_dir)}")
except Exception as e:
    print(f"\nerror while executing the crawler: {e}\n")

In [ ]:

pipeline_dir = os.getcwd()
herbcore_dir = os.path.abspath(os.path.join(pipeline_dir, '..'))
folder_downloader = os.path.join(herbcore_dir, 'herbcore_tool', 'downloader-specieslink-master')
script_download = os.path.normpath(os.path.join(folder_downloader, 'use-dezoomify-rs.py'))
exe_path = os.path.normpath(os.path.join(folder_downloader, 'dezoomify-rs.exe'))

csv = input("inform the name of the csv generated in the previous step: ").strip()
csv_path = os.path.abspath(os.path.join(pipeline_dir, csv))
output_images = input("name for the output folder: ").strip()
output_images_path = os.path.abspath(os.path.join(pipeline_dir, output_images))

if not os.path.exists(csv_path):
    print(f"{csv_path} not found")
else:
    try:
        print(f"\executing images download in '{output_images_path}' folder...\n")
        print("<!!!> ATTENTION: THIS CAN TAKE DAYS <!!!>\n")

        result = subprocess.Popen(
            [sys.executable, script_download, '--input', csv_path, '--output', output_images_path],
            cwd=folder_downloader, 
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            universal_newlines=True
        )

        for line in result.stdout:
            print(line.strip(), flush=True)

        result.wait()

        if result.returncode == 0:
            print(result.stdout)
            print("\nimage download concluded with success\n")
        else:
            print("\nscript error:")
            print(result.stderr)
            print(result.stdout)

    except Exception as e:
        print(f"\nan error occurred while downloading the images: {e}\n")